# BraTS 2021: download, unpack and check the data

This notebook gets the BraTS 2021 brain tumor MRI data onto your machine and proves it is complete and correct.

**What it does, in order**

1. Downloads the Task 1 segmentation training set from the Kaggle mirror: 1,251 patients, each with 4 MRI scans and a tumor mask.
2. Unpacks it.
3. Checks every patient has all 5 files with the right image size, and saves a manifest CSV you can use for training later.
4. Shows one patient's scans and tumor regions, so you can see with your own eyes that the data is real.
5. Optional: the validation set (219 patients, no masks) from Synapse, and the Task 2 MGMT data from Kaggle.

**What it cannot download:** the 570 test cases. They were never released publicly.

**How to run it**

- **VS Code:** install the *Python* and *Jupyter* extensions (both by Microsoft), open this file, click **Select Kernel** (top right) and pick a Python 3.10+ environment. Creating a new `.venv` there is the easiest option and needs no admin rights. Then click **Run All**.
- **Google Colab:** upload the file and click **Runtime > Run all**.

When the notebook needs your Kaggle login, a small input box appears (at the top of the window in VS Code). You only enter it once; it is saved in your user folder.

**Before you start:** you need a Kaggle account and an API key. On kaggle.com go to **Settings > API** and create a token. It gives you a `kaggle.json` file containing your username and key.

**Safe to re-run:** every step checks what is already done and skips it. If the download breaks halfway, just run it again.

## 1. Settings

This is the only cell you should need to change.

In [ ]:
import os
import sys
from pathlib import Path

# ---------------- Change these if you want ----------------
# Where the data goes. Default: a "datasets/brats2021" folder inside your home folder.
# (You can also set the environment variable BRATS_DATA_ROOT instead of editing this.)
DATA_ROOT = Path(os.environ.get("BRATS_DATA_ROOT", Path.home() / "datasets" / "brats2021"))

DOWNLOAD_TASK1_TRAINING = True     # 1,251 patients with tumor masks (the main dataset)
PROCESS_VALIDATION = False         # 219 patients without masks, from Synapse (needs approved access)
SYNAPSE_VALIDATION_ID = ""         # e.g. "syn12345678", copied from the Synapse page of the validation file
DOWNLOAD_TASK2_MGMT = False        # Task 2 MGMT classification data (DICOM format, very large)
SAVE_TO_GOOGLE_DRIVE = False       # Colab only. Free Drive storage (15 GB) is too small for the full set.
LABEL_CHECK_SAMPLE = 25            # patients to fully load for the label check (0 = skip, None = all)
DELETE_ARCHIVES_WHEN_DONE = False  # delete the downloaded .zip/.tar files after unpacking, to free space
MIN_FREE_GB_WARNING = 40           # warn if the disk has less free space than this
# -----------------------------------------------------------

## 2. Install the packages that are missing

Installs only what you don't have yet, into the same Python this notebook runs on.

In [ ]:
import importlib.util
import subprocess
import site

REQUIRED = {  # import name: pip package name
    "kaggle": "kaggle",
    "nibabel": "nibabel",   # reads MRI files (.nii.gz)
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",         # progress bars
}

def pip_install(*packages):
    """Install packages into the Python that runs this notebook."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        # No permission to install for everyone (common on company laptops): install just for you.
        subprocess.check_call(cmd + ["--user"])
        user_site = site.getusersitepackages()
        if user_site not in sys.path:
            sys.path.append(user_site)
    importlib.invalidate_caches()

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing))
    pip_install(*missing)
    print("Done.")
else:
    print("All packages are already installed.")

## 3. Folders and disk space

Creates the folders and tells you how much free space you have. The download is large, and unpacking needs extra room on top of it.

In [ ]:
import shutil

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if SAVE_TO_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_ROOT = Path("/content/drive/MyDrive/brats2021")
    elif "BRATS_DATA_ROOT" not in os.environ:
        DATA_ROOT = Path("/content/brats2021")  # temporary: deleted when the Colab session ends

DOWNLOADS_DIR = DATA_ROOT / "_downloads"               # raw .zip/.tar files
T1_ZIP_DIR    = DOWNLOADS_DIR / "task1_training"
T1_UNZIP_DIR  = DOWNLOADS_DIR / "task1_training_unzipped"
TRAIN_DIR     = DATA_ROOT / "task1_training"           # the unpacked patients end up here
VAL_ZIP_DIR   = DOWNLOADS_DIR / "validation"
VAL_DIR       = DATA_ROOT / "task1_validation"
T2_ZIP_DIR    = DOWNLOADS_DIR / "task2_mgmt"
T2_DIR        = DATA_ROOT / "task2_mgmt"
MANIFEST_CSV  = DATA_ROOT / "manifest_task1_training.csv"

for folder in [T1_ZIP_DIR, T1_UNZIP_DIR, TRAIN_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(DATA_ROOT).free / 1e9
print(f"Running in:  {'Google Colab' if IN_COLAB else 'local Python (VS Code or Jupyter)'}")
print(f"Python:      {sys.version.split()[0]}  ({sys.executable})")
print(f"Data folder: {DATA_ROOT}")
print(f"Free space:  {free_gb:.1f} GB")
if free_gb < MIN_FREE_GB_WARNING:
    print(f"\nWARNING: less than {MIN_FREE_GB_WARNING} GB free. The download or unpacking may fail.")
    print("Change DATA_ROOT in the Settings cell to a drive with more space, then run from the top again.")

## 4. Helper functions

Nothing to change here. These are small tools the next cells use: logging in to Kaggle, unpacking archives, finding patients, and checking files.

In [ ]:
import json
import getpass
import tarfile
import zipfile

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm

MODALITIES = ["flair", "t1", "t1ce", "t2"]   # the 4 MRI scans per patient
EXPECTED_SHAPE = (240, 240, 155)             # every BraTS volume: 240 x 240 pixels, 155 slices

_kaggle_api = None

def get_kaggle_api():
    """Log in to Kaggle. Asks for your username and key only if none are saved yet."""
    global _kaggle_api
    if _kaggle_api is not None:
        return _kaggle_api
    kaggle_json = Path(os.environ.get("KAGGLE_CONFIG_DIR", Path.home() / ".kaggle")) / "kaggle.json"
    has_env = (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")) or os.environ.get("KAGGLE_API_TOKEN")
    access_token_file = kaggle_json.parent / "access_token"   # newer Kaggle token format (KGAT_...)
    if not kaggle_json.exists() and not access_token_file.exists() and not has_env:
        print("No Kaggle login found. Open your kaggle.json (from kaggle.com > Settings > API) and copy the values.")
        username = input("Kaggle username: ").strip()
        key = getpass.getpass("Kaggle API key (hidden while typing): ").strip()
        kaggle_json.parent.mkdir(parents=True, exist_ok=True)
        kaggle_json.write_text(json.dumps({"username": username, "key": key}))
        try:
            os.chmod(kaggle_json, 0o600)
        except OSError:
            pass
        print(f"Saved to {kaggle_json}, so you won't be asked again.")
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    _kaggle_api = api
    return api

def is_valid_zip(path):
    """False for missing or half-downloaded files."""
    return path.exists() and zipfile.is_zipfile(path)

def unzip_all(zip_path, dest):
    with zipfile.ZipFile(zip_path) as zf:
        for member in tqdm(zf.infolist(), desc=f"Unzipping {zip_path.name}", unit="file"):
            zf.extract(member, dest)

def untar_all(tar_path, dest):
    with tarfile.open(tar_path) as tf:
        for member in tqdm(tf.getmembers(), desc=f"Unpacking {tar_path.name}", unit="file"):
            if member.name.startswith(("/", "\\")) or ".." in Path(member.name).parts:
                continue  # never write outside the target folder
            try:
                tf.extract(member, dest, filter="data")
            except TypeError:  # older Python without the 'filter' option
                tf.extract(member, dest)

def unpack_everything(download_dir, staging_dir, dest):
    """Unzip every .zip, then unpack every .tar inside.
    Biggest archive first, so small single-patient archives (corrected cases) are unpacked last and win."""
    for z in sorted(download_dir.glob("*.zip")):
        unzip_all(z, staging_dir)
    tars = set(staging_dir.rglob("*.tar")) | set(download_dir.glob("*.tar"))
    for t in sorted(tars, key=lambda p: p.stat().st_size, reverse=True):
        untar_all(t, dest)

def _landed_on_disk_at(path):
    st = path.stat()
    return getattr(st, "st_birthtime", st.st_ctime)

def find_cases(roots):
    """Find every patient by its FLAIR file and list the paths of its 5 files.
    If a patient appears twice, keep the copy unpacked last (the corrected one)."""
    found = {}
    for root in roots:
        for flair in Path(root).rglob("*_flair.nii*"):
            case_id = flair.name.split("_flair")[0]
            folder = flair.parent
            row = {"case_id": case_id, "folder": str(folder)}
            for part in MODALITIES + ["seg"]:
                matches = sorted(folder.glob(f"{case_id}_{part}.nii*"))
                row[part] = str(matches[0]) if matches else None
            row["_t"] = _landed_on_disk_at(flair)
            if case_id not in found or row["_t"] > found[case_id]["_t"]:
                found[case_id] = row
    rows = sorted(found.values(), key=lambda r: r["case_id"])
    return pd.DataFrame(rows).drop(columns="_t") if rows else pd.DataFrame()

def check_case(row, need_seg=True):
    """Returns '' if the patient is fine, otherwise a short description of the problem."""
    parts = MODALITIES + (["seg"] if need_seg else [])
    has = lambda p: isinstance(row.get(p), str) and row.get(p) != ""
    problems = [f"missing {p}" for p in parts if not has(p)]
    shapes = set()
    for p in parts:
        if has(p):
            try:
                shapes.add(tuple(nib.load(row[p]).shape))  # reads only the file header: fast
            except Exception:
                problems.append(f"{p} unreadable")
    if shapes and shapes != {EXPECTED_SHAPE}:
        problems.append(f"unexpected shape {sorted(shapes)}")
    return "; ".join(problems)

def load_volume(path):
    return np.asanyarray(nib.load(path).dataobj)

def labels_in(seg_path):
    return set(np.unique(load_volume(seg_path)).astype(int).tolist())

def brats_regions(seg):
    """BraTS scores 3 nested regions, not single labels.
    Labels: 1 = necrotic core, 2 = edema, 4 = enhancing tumor (BraTS 2023 renamed 4 to 3)."""
    return {
        "WT (whole tumor)":     np.isin(seg, [1, 2, 3, 4]),
        "TC (tumor core)":      np.isin(seg, [1, 3, 4]),
        "ET (enhancing tumor)": np.isin(seg, [3, 4]),
    }

print("Helpers ready.")

## 5. Download the Task 1 training set

Source: the Kaggle mirror `dschettler8845/brats-2021-task1`. This is the big one and can take a long time. Leave the laptop plugged in and awake. If it stops halfway, run this cell again.

In [ ]:
T1_DATASET = "dschettler8845/brats-2021-task1"
T1_DONE = TRAIN_DIR / ".unpacked_ok"

if not DOWNLOAD_TASK1_TRAINING:
    print("Skipped (DOWNLOAD_TASK1_TRAINING = False).")
elif T1_DONE.exists():
    print("Already downloaded and unpacked. Skipping.")
else:
    zips = sorted(T1_ZIP_DIR.glob("*.zip"))
    if zips and all(is_valid_zip(z) for z in zips):
        print("Download already here:", ", ".join(z.name for z in zips))
    else:
        for z in zips:
            z.unlink()  # remove a broken or half-finished download
        print(f"Downloading {T1_DATASET} from Kaggle...")
        get_kaggle_api().dataset_download_files(T1_DATASET, path=str(T1_ZIP_DIR), unzip=False, quiet=False)
        print("Download finished.")

## 6. Unpack it

The download is a `.zip` that contains `.tar` archives, which contain the patient folders. This cell opens both layers.

In [ ]:
if not DOWNLOAD_TASK1_TRAINING or T1_DONE.exists():
    print("Nothing to unpack.")
else:
    if not list(T1_ZIP_DIR.glob("*.zip")):
        raise FileNotFoundError("No download found. Run the download cell first.")
    unpack_everything(T1_ZIP_DIR, T1_UNZIP_DIR, TRAIN_DIR)
    T1_DONE.write_text("ok")
    print("Unpacked into", TRAIN_DIR)

## 7. Check every patient and save a manifest

For each patient this checks that all 5 files exist (FLAIR, T1, T1ce, T2 and the tumor mask) and that every image is 240 x 240 x 155.

The result is saved as a CSV "manifest": one row per patient with the path of each file. Later, your training code can read this CSV instead of searching folders.

In [ ]:
cases = find_cases([TRAIN_DIR, T1_UNZIP_DIR])
if cases.empty:
    raise FileNotFoundError(f"No patients found under {TRAIN_DIR}. Run the download and unpack cells first.")

cases["problems"] = [check_case(r) for r in tqdm(cases.to_dict("records"), desc="Checking patients", unit="patient")]
cases["complete"] = cases["problems"] == ""
cases.to_csv(MANIFEST_CSV, index=False)

n_total, n_ok = len(cases), int(cases["complete"].sum())
print(f"Patients found:  {n_total}  (expected 1,251)")
print(f"Complete and OK: {n_ok}")
print(f"Manifest saved:  {MANIFEST_CSV}")
if n_ok < n_total:
    print("\nPatients with problems:")
    display(cases.loc[~cases["complete"], ["case_id", "problems"]])
cases.head()

## 8. Check the tumor labels

Each voxel in a tumor mask has a number:

| Label | Meaning |
|---|---|
| 0 | healthy tissue or background |
| 1 | necrotic tumor core (dead tissue in the middle) |
| 2 | edema (swelling around the tumor) |
| 4 | enhancing tumor (the active part that lights up with contrast) |

There is no label 3 in BraTS 2021. That matters for training: with 4 classes, label 4 must be changed to 3 first, otherwise the one-hot step crashes. This cell loads a sample of masks and confirms which labels are really there.

In [ ]:
ok_cases = cases[cases["complete"]]
if LABEL_CHECK_SAMPLE == 0 or ok_cases.empty:
    print("Label check skipped.")
else:
    sample = ok_cases if LABEL_CHECK_SAMPLE is None else ok_cases.sample(min(LABEL_CHECK_SAMPLE, len(ok_cases)), random_state=0)
    found_labels = set()
    for seg_path in tqdm(sample["seg"], desc="Reading masks", unit="mask"):
        found_labels |= labels_in(seg_path)
    print(f"Labels found in {len(sample)} masks: {sorted(found_labels)}")
    if found_labels <= {0, 1, 2, 4}:
        print("As expected for BraTS 2021.")
        print("Before training with 4 classes, remap:  seg[seg == 4] = 3")
    elif 3 in found_labels and 4 not in found_labels:
        print("This looks like BraTS 2023 naming (enhancing tumor = 3). No remap needed.")
    else:
        print("Unexpected labels. Look at these files before training.")

## 9. Look at one patient

The 4 scans are like 4 photos of the same room taken under different lighting: each one makes different tissue stand out. The last panel puts the tumor mask on top of the FLAIR scan.

The slice shown is the one with the most tumor in it. To look at a different patient, set `CASE_ID`, for example `"BraTS2021_00005"`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

CASE_ID = None  # None = first complete patient

if ok_cases.empty:
    raise ValueError("No complete patients to show.")
row = ok_cases.iloc[0] if CASE_ID is None else cases[cases["case_id"] == CASE_ID].iloc[0]

vols = {m: load_volume(row[m]).astype(np.float32) for m in MODALITIES}
seg = load_volume(row["seg"]).astype(np.int16)

tumor_per_slice = (seg > 0).sum(axis=(0, 1))
z = int(tumor_per_slice.argmax()) if tumor_per_slice.max() > 0 else seg.shape[2] // 2

def axial(volume):
    return np.rot90(volume[:, :, z])  # rotate so the head points up

label_view = np.zeros_like(seg)
label_view[seg == 1] = 1
label_view[seg == 2] = 2
label_view[np.isin(seg, [3, 4])] = 3
label_colors = ListedColormap(["#e41a1c", "#4daf4a", "#ffd92f"])

titles = {"flair": "FLAIR", "t1": "T1", "t1ce": "T1ce (with contrast)", "t2": "T2"}
fig, axes = plt.subplots(1, 5, figsize=(20, 4.6))
for ax, m in zip(axes, MODALITIES):
    ax.imshow(axial(vols[m]), cmap="gray")
    ax.set_title(titles[m])
axes[4].imshow(axial(vols["flair"]), cmap="gray")
axes[4].imshow(np.ma.masked_equal(axial(label_view), 0), cmap=label_colors, vmin=1, vmax=3,
               alpha=0.6, interpolation="nearest")
axes[4].set_title("Tumor mask on FLAIR")
axes[4].legend(handles=[Patch(color="#e41a1c", label="1 necrotic core"),
                        Patch(color="#4daf4a", label="2 edema"),
                        Patch(color="#ffd92f", label="4 enhancing tumor")],
               loc="lower right", fontsize=8)
for ax in axes:
    ax.axis("off")
fig.suptitle(f"{row['case_id']}, axial slice {z} of {seg.shape[2]}")
plt.tight_layout()
plt.show()

## 10. The three tumor regions BraTS actually scores

Challenge scores are not per label. They are per **region**, and the regions sit inside each other like three nested circles:

- **WT, whole tumor:** everything (labels 1 + 2 + 4). The biggest circle.
- **TC, tumor core:** the tumor without the swelling (labels 1 + 4).
- **ET, enhancing tumor:** only the active part (label 4). The smallest.

Each voxel is 1 mm x 1 mm x 1 mm, so counting voxels and dividing by 1,000 gives the volume in cm³.

In [ ]:
regions = brats_regions(seg)
for name, mask in regions.items():
    print(f"{name:<22} {mask.sum() / 1000:7.1f} cm³")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, (name, mask) in zip(axes, regions.items()):
    ax.imshow(axial(vols["flair"]), cmap="gray")
    ax.imshow(np.ma.masked_equal(axial(mask.astype(np.uint8)), 0), cmap=ListedColormap(["#ff7f00"]),
              alpha=0.6, interpolation="nearest")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Optional: validation set from Synapse (219 patients, no masks)

Only runs if `PROCESS_VALIDATION = True` in Settings.

The validation set is only on Synapse, and you need approved access first:

1. Create an account on synapse.org and request access to the BraTS 2021 data from the challenge page.
2. Once approved, either download the validation file by hand and put it in the `_downloads/validation` folder, **or** paste its Synapse ID (starts with `syn`) into `SYNAPSE_VALIDATION_ID` and let this cell download it. For that you need a Synapse personal access token (synapse.org > Account Settings > Personal Access Tokens).

These patients have no masks, so you can only score predictions on them by uploading to the challenge platform.

In [ ]:
VAL_DONE = VAL_DIR / ".unpacked_ok"

if not PROCESS_VALIDATION:
    print("Skipped (PROCESS_VALIDATION = False).")
else:
    VAL_ZIP_DIR.mkdir(parents=True, exist_ok=True)
    VAL_DIR.mkdir(parents=True, exist_ok=True)
    archives = list(VAL_ZIP_DIR.glob("*.zip")) + list(VAL_ZIP_DIR.glob("*.tar"))

    if not archives and SYNAPSE_VALIDATION_ID and not VAL_DONE.exists():
        if importlib.util.find_spec("synapseclient") is None:
            pip_install("synapseclient")
        import synapseclient
        token = os.environ.get("SYNAPSE_AUTH_TOKEN") or getpass.getpass("Synapse personal access token (hidden): ")
        syn = synapseclient.Synapse()
        syn.login(authToken=token, silent=True)
        print(f"Downloading {SYNAPSE_VALIDATION_ID} from Synapse...")
        syn.get(SYNAPSE_VALIDATION_ID, downloadLocation=str(VAL_ZIP_DIR))
        archives = list(VAL_ZIP_DIR.glob("*.zip")) + list(VAL_ZIP_DIR.glob("*.tar"))

    if not archives and not VAL_DONE.exists():
        print(f"No validation file yet. Put the file from Synapse in:\n  {VAL_ZIP_DIR}\n"
              "or set SYNAPSE_VALIDATION_ID in Settings, then run this cell again.")
    else:
        if not VAL_DONE.exists():
            unpack_everything(VAL_ZIP_DIR, VAL_DIR, VAL_DIR)
            VAL_DONE.write_text("ok")
        val_cases = find_cases([VAL_DIR])
        val_cases["problems"] = [check_case(r, need_seg=False) for r in val_cases.to_dict("records")]
        val_cases["complete"] = val_cases["problems"] == ""
        val_csv = DATA_ROOT / "manifest_task1_validation.csv"
        val_cases.to_csv(val_csv, index=False)
        print(f"Validation patients found: {len(val_cases)}  (expected 219)")
        print(f"Complete and OK:           {int(val_cases['complete'].sum())}")
        print(f"Manifest saved:            {val_csv}")

## 12. Optional: Task 2, MGMT classification data (Kaggle competition)

Only runs if `DOWNLOAD_TASK2_MGMT = True` in Settings.

This is a different task: predicting one yes/no value per patient (MGMT methylation) instead of drawing the tumor. The scans are in DICOM format, not `.nii.gz`, and the download is very large, so check your disk first.

Kaggle only lets you download competition data after you accept its rules once on the website. If this cell says the download was refused, open the link it prints, accept the rules, and run it again.

In [ ]:
T2_COMPETITION = "rsna-miccai-brain-tumor-radiogenomic-classification"
T2_DONE = T2_DIR / ".unpacked_ok"

if not DOWNLOAD_TASK2_MGMT:
    print("Skipped (DOWNLOAD_TASK2_MGMT = False).")
else:
    T2_ZIP_DIR.mkdir(parents=True, exist_ok=True)
    T2_DIR.mkdir(parents=True, exist_ok=True)
    if not T2_DONE.exists():
        zips = sorted(T2_ZIP_DIR.glob("*.zip"))
        if not (zips and all(is_valid_zip(z) for z in zips)):
            for z in zips:
                z.unlink()
            try:
                get_kaggle_api().competition_download_files(T2_COMPETITION, path=str(T2_ZIP_DIR), quiet=False)
            except Exception as e:
                raise RuntimeError(
                    "Kaggle refused the download. Accept the competition rules here, then run this cell again:\n"
                    f"https://www.kaggle.com/competitions/{T2_COMPETITION}/rules\n({e})")
        for z in sorted(T2_ZIP_DIR.glob("*.zip")):
            unzip_all(z, T2_DIR)
        T2_DONE.write_text("ok")
    labels_csv = next(T2_DIR.rglob("train_labels.csv"), None)
    if labels_csv is not None:
        mgmt = pd.read_csv(labels_csv)
        print(f"Training patients with an MGMT label: {len(mgmt)}")
        if "MGMT_value" in mgmt.columns:
            print(mgmt["MGMT_value"].value_counts().rename({0: "0 = unmethylated", 1: "1 = methylated"}))
        display(mgmt.head())

## 13. Optional: free up disk space

With `DELETE_ARCHIVES_WHEN_DONE = True`, this deletes the downloaded `.zip` and `.tar` files, but only for parts that finished unpacking successfully. The unpacked patient folders stay. The catch: if you ever need to unpack again, you have to download again.

In [ ]:
if not DELETE_ARCHIVES_WHEN_DONE:
    print("Keeping the archives. Set DELETE_ARCHIVES_WHEN_DONE = True and run this cell again to free space.")
else:
    to_delete = []
    if T1_DONE.exists():
        to_delete += list(T1_ZIP_DIR.glob("*.zip")) + list(T1_UNZIP_DIR.rglob("*.tar"))
    if (VAL_DIR / ".unpacked_ok").exists():
        to_delete += list(VAL_ZIP_DIR.glob("*.zip")) + list(VAL_ZIP_DIR.glob("*.tar")) + list(VAL_DIR.rglob("*.tar"))
    if (T2_DIR / ".unpacked_ok").exists():
        to_delete += list(T2_ZIP_DIR.glob("*.zip"))
    freed = 0
    for f in set(to_delete):
        freed += f.stat().st_size
        f.unlink()
    print(f"Deleted {len(set(to_delete))} archive files, freed {freed / 1e9:.1f} GB.")

## 14. Summary

In [ ]:
print("BraTS 2021 data")
print(f"  Folder:            {DATA_ROOT}")
print(f"  Training patients: {int(cases['complete'].sum())} complete of {len(cases)}  ->  {TRAIN_DIR}")
print(f"  Training manifest: {MANIFEST_CSV}")
if (VAL_DIR / ".unpacked_ok").exists():
    print(f"  Validation:        {VAL_DIR}")
if (T2_DIR / ".unpacked_ok").exists():
    print(f"  Task 2 (MGMT):     {T2_DIR}")
print(f"  Free space left:   {shutil.disk_usage(DATA_ROOT).free / 1e9:.1f} GB")

### Using the data in your own code

The manifest makes loading simple. For example:

```python
import pandas as pd
import nibabel as nib

cases = pd.read_csv("path/to/manifest_task1_training.csv")
cases = cases[cases["complete"]]

row = cases.iloc[0]
flair = nib.load(row["flair"]).get_fdata()   # shape (240, 240, 155)
seg = nib.load(row["seg"]).get_fdata()
seg[seg == 4] = 3                            # remap before one-hot with 4 classes
```

Two reminders for training:

- Split train and validation by **patient** (by row of this manifest), never by slice, or the scores will be misleadingly high.
- Score with the nested regions (WT, TC, ET) from section 10, not with single labels.